# Build Filtered Datasets

`dataset/final_dataset.csv`를 기준으로 전처리된 CSV 두 개를 같은 `dataset/` 폴더에 생성한다.

- `final_dataset_filtered.csv`: binary 학습용 filtered dataset
- `final_dataset_filtered_3class.csv`: 낙상 포함 video만 남기고 `label_3class`를 추가한 3-class 학습용 dataset

전처리 본체는 `scripts/build_filtered_dataset.py`를 그대로 사용한다.

In [ ]:
# Runtime setup
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

# 필요하면 직접 지정하세요. None이면 후보 경로에서 자동 탐색합니다.
PROJECT_ROOT_OVERRIDE = None  # 예: Path('/content/drive/MyDrive/Falling-Model-Development')

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path('/content/drive/MyDrive/Graduate-Project/Falling-Model-Development'),
    Path('/content/drive/MyDrive/Falling-Model-Development'),
    Path('/content/drive/MyDrive/졸업 과제/Falling-Detection-Development'),
]

def find_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    for candidate in PROJECT_ROOT_CANDIDATES:
        if (candidate / 'scripts' / 'build_filtered_dataset.py').exists() and (candidate / 'dataset' / 'final_dataset.csv').exists():
            return candidate.resolve()
    raise FileNotFoundError('Project root not found. Set PROJECT_ROOT_OVERRIDE to the repo path on Drive.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'DATASET_DIR  = {PROJECT_ROOT / "dataset"}')

In [ ]:
# Paths and preprocessing controls
DATASET_DIR = PROJECT_ROOT / 'dataset'
INPUT_CSV = DATASET_DIR / 'final_dataset.csv'
FILTERED_CSV = DATASET_DIR / 'final_dataset_filtered.csv'
FILTERED_3CLASS_CSV = DATASET_DIR / 'final_dataset_filtered_3class.csv'
TMP_FULL_CSV = DATASET_DIR / 'final_dataset_filtered_with_3class_tmp.csv'

OVERWRITE = True
KEEP_INTERMEDIATE = False

# True면 기존 final_dataset_filtered_3class.csv와 같은 방식으로 낙상 포함 video만 저장합니다.
THREE_CLASS_ONLY_FALL_VIDEOS = True

FILTER_ARGS = {
    'min_cutoff': 0.5,
    'beta': 0.3,
    'd_cutoff': 1.0,
    'conf_alpha': 0.5,
    'conf_thr': 0.15,
    'ema_deriv_alpha': 0.4,
}

for path in [INPUT_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)

if not OVERWRITE:
    existing = [p for p in [FILTERED_CSV, FILTERED_3CLASS_CSV, TMP_FULL_CSV] if p.exists()]
    if existing:
        raise FileExistsError(f'Output already exists: {existing}')

print(f'input              : {INPUT_CSV}')
print(f'filtered output    : {FILTERED_CSV}')
print(f'3-class output     : {FILTERED_3CLASS_CSV}')

In [ ]:
# Run Pipeline D preprocessing through the repo script
cmd = [
    sys.executable,
    'scripts/build_filtered_dataset.py',
    '--input', str(INPUT_CSV),
    '--output', str(TMP_FULL_CSV),
    '--min-cutoff', str(FILTER_ARGS['min_cutoff']),
    '--beta', str(FILTER_ARGS['beta']),
    '--d-cutoff', str(FILTER_ARGS['d_cutoff']),
    '--conf-alpha', str(FILTER_ARGS['conf_alpha']),
    '--conf-thr', str(FILTER_ARGS['conf_thr']),
    '--ema-deriv-alpha', str(FILTER_ARGS['ema_deriv_alpha']),
]

print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Split the script output into binary filtered and 3-class filtered CSVs
import pandas as pd

df = pd.read_csv(TMP_FULL_CSV, low_memory=False)

required = {'video_id', 'time_sec', 'label'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

if 'label_3class' not in df.columns:
    df['label_3class'] = df['label'].copy()
    for _, grp in df.groupby('video_id', sort=False):
        fall_idx = grp.index[grp['label'] == 1]
        if len(fall_idx) > 0:
            last_fall = fall_idx.max()
            df.loc[grp.index[grp.index > last_fall], 'label_3class'] = 2

binary_df = df.drop(columns=['label_3class'])
binary_df.to_csv(FILTERED_CSV, index=False)

if THREE_CLASS_ONLY_FALL_VIDEOS:
    fall_video_ids = df.loc[df['label'] == 1, 'video_id'].drop_duplicates()
    class_df = df[df['video_id'].isin(fall_video_ids)].copy()
else:
    class_df = df.copy()

class_df.to_csv(FILTERED_3CLASS_CSV, index=False)

if not KEEP_INTERMEDIATE:
    TMP_FULL_CSV.unlink(missing_ok=True)

print(f'wrote {FILTERED_CSV} rows={len(binary_df):,} cols={len(binary_df.columns)}')
print(f'wrote {FILTERED_3CLASS_CSV} rows={len(class_df):,} cols={len(class_df.columns)}')
print(f'label: {binary_df["label"].value_counts().sort_index().to_dict()}')
print(f'label_3class: {class_df["label_3class"].value_counts().sort_index().to_dict()}')

In [ ]:
# Quick validation
for path in [FILTERED_CSV, FILTERED_3CLASS_CSV]:
    header = pd.read_csv(path, nrows=0).columns.tolist()
    size_gb = path.stat().st_size / (1024 ** 3)
    print(f'{path.name}: {len(header)} columns, {size_gb:.2f} GiB')
    print(header[-8:])